In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Data Cleaning

When using data for your projects you'll have to deal with the reality that the humans making said data are not perfect. Some things you could deal with:
- Wrong data types
- Null/Empty values
- Wrong Delimeters
- Duplicate rows
- Outliers

For Time Series data we specifically need to deal with lookahead bias.

In [4]:
df = pd.read_csv("data/messy_sales_data.csv")
df.head(10)

ParserError: Error tokenizing data. C error: Expected 14 fields in line 4, saw 15


Ok so the error was weird, it occurs because pandas.read_csv() tries to load a file and Pandas finds more columns in a specific row than it expected based on the header or the very first row. So we have a few options:
1. Skip the problematic row
2. Could be a wrong delimiter/separator
3. Some columns could be missing
Since this occured in line 4, we know that it isn't a missing column. We can try getting rid of it and if it becomes a recurring issue then we can check delimeters

In [6]:
df = pd.read_csv("data/messy_sales_data.csv", encoding='utf-8', on_bad_lines='skip')   # change delimeters with sep=';', sep=',', and so on
df.head(10)

,OrderID,Customer Name,Product,Category,Quantity,Unit Price,Total Amount,Order Date,City,Payment Method,Status,Salesperson,Notes
ORD-1409-45,Johnson,Sarah,Mobile Phone,Gadgets,5,872.75,4363.75,2022-01-31,MIA,cash,Completed,SP-833,URGENT
ORD-5557-10,Online Customer,Smartphone,ELECTRONICS,2,1538.81,3077.62,2022-04-15,Chicago,IL,Google Pay,NaN,SP-144,bulk order
ORD-2169-87,R Wilson,Smartphone,Computers,4,1854.51,7418.04,12-06-23,CHI,Credit Card,Refunded,SP-924,URGENT,NaN
ORD-6155-37,R Wilson,Wireless Headphones,ELECTRONICS,4,322.28,1289.12,07-29-23,Seattle,Bank Transfer,NaN,SP-324,bulk order,NaN
ORD-1771-24,JOHN SMITH,Mechanical Keyboard,Gadgets,4,800.3,3201.2,2023/09/03,Miami,FL,Credit Card,Pending,SP-649,discount applied
ORD-8123-30,E Davis,Laptop,Mobile,2,257.5,515.0,03-10-2023,MIA,cash,Refunded,SP-880,corporate client,NaN
ORD-6310-72,John Smith,Lpt,Accessories,3,519.7,1559.1,2023-08-04,Miami,Debit Card,CANCEL,SP-231,URGENT,NaN
ORD-7932-37,Robert Wilson,Wireless Mouse,Computers,4,1317.26,5269.04,03-25-23,L.A.,PayPal,Pending,SP-702,NaN,NaN
ORD-1964-39,Smith,John,laptop,Accessories,2,996.56,1993.12,07/07/2023,Seattle,cash,pending,SP-584,NaN
ORD-6804-64,Emily Davis,Earphones,Electronics,1,1470.02,1470.02,04/22/2022,Miami,FL,CC,Cancelled,SP-385,NaN


Weird, seems some things are off..

You'll notice some items are in the wrong columns, we should check to make sure the delimeter is correct for all entries

In [12]:
# check raw data
with open("data/messy_sales_data.csv", 'rb') as f:
    raw_data = f.read()
raw_data[:600]

b'OrderID,Customer Name,Product,Category,Quantity,Unit Price,Total Amount,Order Date,City,Payment Method,Status,Salesperson,Notes\nORD-1409-45,Johnson, Sarah,Mobile Phone,Gadgets,5,872.75,4363.75,2022-01-31,MIA,cash,Completed,SP-833,URGENT\nORD-5557-10,Online Customer,Smartphone,ELECTRONICS,2,1538.81,3077.62,2022-04-15,Chicago, IL,Google Pay,,SP-144,bulk order\nORD-5803-90,Wilson, Robert,Tablet Computer,Computers,1,1557.48,1557.48,Mar 23, 2022,NYC,Bank Transfer,Shipped,SP-954,URGENT\nORD-2169-87,R Wilson,Smartphone,Computers,4,1854.51,7418.04,12-06-23,CHI,Credit Card,Refunded,SP-924,URGENT\nORD-6155-'

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 43092 entries, ORD-1409-45 to ORD-8816-21
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   OrderID         42580 non-null  object
 1   Customer Name   42254 non-null  object
 2   Product         42171 non-null  object
 3   Category        42113 non-null  object
 4   Quantity        42126 non-null  object
 5   Unit Price      42156 non-null  object
 6   Total Amount    42166 non-null  object
 7   Order Date      39474 non-null  object
 8   City            41034 non-null  object
 9   Payment Method  39665 non-null  object
 10  Status          36700 non-null  object
 11  Salesperson     34680 non-null  object
 12  Notes           13717 non-null  object
dtypes: object(13)
memory usage: 4.6+ MB


In [17]:
# Check where the missing values are
print(df.isnull().sum())

# drop all rows where there is a NaN value
df_cleaned = df.dropna()

df_cleaned.head(10)

OrderID             512
Customer Name       838
Product             921
Category            979
Quantity            966
Unit Price          936
Total Amount        926
Order Date         3618
City               2058
Payment Method     3427
Status             6392
Salesperson        8412
Notes             29375
dtype: int64


,OrderID,Customer Name,Product,Category,Quantity,Unit Price,Total Amount,Order Date,City,Payment Method,Status,Salesperson,Notes
ORD-1409-45,Johnson,Sarah,Mobile Phone,Gadgets,5,872.75,4363.75,2022-01-31,MIA,cash,Completed,SP-833,URGENT
ORD-1771-24,JOHN SMITH,Mechanical Keyboard,Gadgets,4,800.3,3201.2,2023/09/03,Miami,FL,Credit Card,Pending,SP-649,discount applied
ORD-4853-61,John S.,Display,Computers,1,867.49,867.49,08-21-23,Los Angeles,CA,PayPal,Delivered,SP-505,bulk order
ORD-2188-11,E Davis,Keyboard,Tech,2,308.29,616.58,2022-12-24,Chicago,IL,Apple Pay,Cancelled,SP-656,corporate client
ORD-2988-68,JOHN SMITH,Earphones,Accessories,4,1452.58,5810.32,2023/06/01,Miami,FL,CC,Cancelled,SP-586,discount applied
ORD-7624-31,E Davis,Laptop Computer,Electronics,5,102.85,514.25,2023/10/21,Houston,TX,Cash,Completed,SP-435,bulk order
ORD-9031-23,Emily Davis,Tablet Computer,electronics,4,1067.41,4269.64,29-10-2023,Miami,FL,CC,pending,SP-848,URGENT
ORD-4130-20,Johnson,Sarah,Computer Mouse,ELECTRONICS,2,1309.51,2619.02,2023/05/18,New York City,Apple Pay,Refunded,SP-349,bulk order
ORD-3564-10,Emily Davis,Earphones,electronics,2,601.3,1202.6,2023-04-10,Los Angeles,CA,cash,Refunded,SP-657,discount applied
ORD-9117-46,Unknown,Headphones,Electronics,3,276.05,828.15,Feb 18,2023,New York,CASH,pending,SP-292,bulk order


In [18]:
# drop rows where only certain columns are nans
df_cleaned = df.dropna(subset=['Salesperson', "Quantity", "Order Date"])

# Add specific values for NaNs
df['Salesperson'] = df['Salesperson'].fillna('Unknown')

# fill in value with a guess (mean or median)
df['Unit Price'] = df['Unit Price'].fillna(df['Unit Price'].median())

TypeError: could not convert string to float: 'negotiable'

In [25]:
# Well looks like we have to deal with datatype issues now...

# Convert a column to numeric, forcing errors to NaN
df['Unit Price'] = pd.to_numeric(df['Unit Price'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

df['Order Date'] = pd.to_datetime(df['Order Date'], format='%Y-%m-%d', errors='coerce')

In [26]:
# Check how many duplicate rows exist
print(df.duplicated().sum())

# Drop exact duplicate rows (keeps the first occurrence by default)
df = df.drop_duplicates()

0


In [28]:
# Get rid of outleiers

# Calculate Q1 (25th percentile) and Q3 (75th percentile)
Q1 = df['Quantity'].quantile(0.25)
Q3 = df['Quantity'].quantile(0.75)
IQR = Q3 - Q1

# Define the acceptable bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter the dataframe to keep only data within the bounds
df_no_outliers = df[(df['Quantity'] >= lower_bound) & (df['Quantity'] <= upper_bound)]

# Exploratory Data Analysis (EDA)

## Quant Specific